In [1]:
import pandas as pd
import yfinance as yf
import ta
import pandas_datareader.data as web
import numpy as np
import torch
import quantstats as qs
import os
import json
import warnings

warnings.filterwarnings('ignore')

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback

In [2]:
# ==========================================
# 0. HARDWARE SETUP
# ==========================================
device = "cpu"
print(f"✅ Device Strategy: '{device}' เพราะ PPO+MlpPolicy เป็นโมเดลเล็ก การใช้ GPU จะช้ากว่าเพราะ overhead การส่งข้อมูล")
if torch.cuda.is_available():
    print(f"   - พบ GPU: {torch.cuda.get_device_name(0)} (แต่ไม่แนะนำให้ใช้กับ PPO MlpPolicy)")
else:
    print("   - ไม่พบการ์ดจอแยก ระบบจะทำงานบน CPU (เหมาะสมอยู่แล้วสำหรับโปรเจกต์นี้)")

✅ Device Strategy: 'cpu' เพราะ PPO+MlpPolicy เป็นโมเดลเล็ก การใช้ GPU จะช้ากว่าเพราะ overhead การส่งข้อมูล
   - พบ GPU: NVIDIA GeForce RTX 5060 Ti (แต่ไม่แนะนำให้ใช้กับ PPO MlpPolicy)


In [3]:
# ==========================================
# 1. GLOBAL CONFIGURATION
# ==========================================
TICKERS = ['DIA', 'QQQ', 'SPY']
TRANSACTION_COST_PCT = 0.015  
FED_FUNDS_LAG_DAYS = 30   
M2_LAG_DAYS = 60 

USE_MULTI_TIMEFRAME = False

# 🔧 แก้ไข #3 (Latent Bug): เอา Indicator ที่เป็น "ราคาดิบ" (Raw Price) เช่น EMA_200, MACD_line ออกจาก State Space 
# เพื่อป้องกัน Neural Network สับสนกับสเกลตัวเลขที่ต่างกันมาก (เก็บไว้เฉพาะค่าที่เป็น Ratio และ Normalize แล้ว)
COMPLETE_TECHNICAL_INDICATORS = [
    'RSI_14', 'RSI_signal', 'RSI_overbought', 'RSI_oversold', 'RSI_above_center',
    'MACD_histogram', 'MACD_cross',
    'price_to_EMA50_ratio', 'price_to_EMA200_ratio',
    'EMA_golden_cross', 'EMA_death_cross',
    'StochRSI_K', 'StochRSI_D', 'StochRSI_cross',
]

MULTI_TIMEFRAME_INDICATORS = [
    'RSI_weekly', 'RSI_monthly',
    'MACD_histogram_weekly', 'MACD_histogram_monthly',
]

MACRO_INDICATORS = ['vix', 'bond_yield', 'gold', 'wti', 'fed_rate', 'm2']

TECHNICAL_INDICATORS = COMPLETE_TECHNICAL_INDICATORS + (
    MULTI_TIMEFRAME_INDICATORS if USE_MULTI_TIMEFRAME else []
)
FEATURES = TECHNICAL_INDICATORS + MACRO_INDICATORS

In [4]:
# ==========================================
# 2. INDICATOR COMPUTATION 
# ==========================================
def compute_complete_indicators(df, price_col='close'):
    close = df[price_col]

    rsi = ta.momentum.RSIIndicator(close=close, window=14).rsi()
    df['RSI_14'] = rsi
    df['RSI_signal'] = rsi.rolling(window=9).mean()
    df['RSI_overbought'] = (rsi > 70).astype(float)
    df['RSI_oversold'] = (rsi < 30).astype(float)
    df['RSI_above_center'] = (rsi > 50).astype(float)

    macd_obj = ta.trend.MACD(close=close, window_slow=26, window_fast=12, window_sign=9)
    macd_line = macd_obj.macd()
    macd_signal = macd_obj.macd_signal()
    # ยังคงคำนวณและเก็บลง df เพื่อใช้เป็นตัวแปรตั้งต้น แต่จะไม่ถูกดึงไปใช้ใน State Space
    df['MACD_line'] = macd_line
    df['MACD_signal'] = macd_signal
    df['MACD_histogram'] = macd_obj.macd_diff()
    prev_diff = (macd_line - macd_signal).shift(1)
    curr_diff = macd_line - macd_signal
    df['MACD_cross'] = np.select(
        [(prev_diff <= 0) & (curr_diff > 0), (prev_diff >= 0) & (curr_diff < 0)],
        [1.0, -1.0], default=0.0
    )

    ema12 = ta.trend.EMAIndicator(close=close, window=12).ema_indicator()
    ema26 = ta.trend.EMAIndicator(close=close, window=26).ema_indicator()
    ema50 = ta.trend.EMAIndicator(close=close, window=50).ema_indicator()
    ema200 = ta.trend.EMAIndicator(close=close, window=200).ema_indicator()
    
    df['EMA_12'] = ema12
    df['EMA_26'] = ema26
    df['EMA_50'] = ema50
    df['EMA_200'] = ema200
    df['price_to_EMA50_ratio'] = (close - ema50) / ema50
    df['price_to_EMA200_ratio'] = (close - ema200) / ema200
    prev_ema_diff = (ema50 - ema200).shift(1)
    curr_ema_diff = ema50 - ema200
    df['EMA_golden_cross'] = ((prev_ema_diff <= 0) & (curr_ema_diff > 0)).astype(float)
    df['EMA_death_cross'] = ((prev_ema_diff >= 0) & (curr_ema_diff < 0)).astype(float)

    stochrsi_obj = ta.momentum.StochRSIIndicator(close=close, window=14, smooth1=3, smooth2=3)
    stoch_k = stochrsi_obj.stochrsi_k()
    stoch_d = stochrsi_obj.stochrsi_d()
    df['StochRSI_K'] = stoch_k
    df['StochRSI_D'] = stoch_d
    prev_stoch_diff = (stoch_k - stoch_d).shift(1)
    curr_stoch_diff = stoch_k - stoch_d
    bullish_cross = (prev_stoch_diff <= 0) & (curr_stoch_diff > 0) & (stoch_k < 0.2)
    bearish_cross = (prev_stoch_diff >= 0) & (curr_stoch_diff < 0) & (stoch_k > 0.8)
    df['StochRSI_cross'] = np.select([bullish_cross, bearish_cross], [1.0, -1.0], default=0.0)

    return df

def compute_multi_timeframe_features(df, price_col='close', date_col='date'):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    close = df[price_col]

    weekly_close = close.resample('W-FRI').last()
    weekly_rsi = ta.momentum.RSIIndicator(close=weekly_close, window=14).rsi().shift(1)
    df['RSI_weekly'] = weekly_rsi.reindex(df.index, method='ffill')

    weekly_macd_hist = ta.trend.MACD(close=weekly_close).macd_diff().shift(1)
    df['MACD_histogram_weekly'] = weekly_macd_hist.reindex(df.index, method='ffill')

    monthly_close = close.resample('ME').last()
    monthly_rsi = ta.momentum.RSIIndicator(close=monthly_close, window=14).rsi().shift(1)
    df['RSI_monthly'] = monthly_rsi.reindex(df.index, method='ffill')

    monthly_macd_hist = ta.trend.MACD(close=monthly_close).macd_diff().shift(1)
    df['MACD_histogram_monthly'] = monthly_macd_hist.reindex(df.index, method='ffill')

    df = df.reset_index()
    return df

In [5]:
# ==========================================
# 3. HIGH-PERFORMANCE DATA PIPELINE
# ==========================================
# 🔧 แก้ไข #4 (Latent Bug): ปรับ warmup_calendar_days เป็น 850 เผื่อการใช้งาน MACD_monthly ในอนาคต
def fetch_and_prepare_data(tickers, start_date, end_date, warmup_calendar_days=850):
    buffer_start = (pd.to_datetime(start_date) - pd.Timedelta(days=warmup_calendar_days)).strftime('%Y-%m-%d')
    print(f"📦 กําลังดาวน์โหลดข้อมูลจาก {start_date} ถึง {end_date} (ดึงเผื่อ Buffer ตั้งแต่ {buffer_start})...")
    processed_dfs = []

    for ticker in tickers:
        try:
            df = yf.download(ticker, start=buffer_start, end=end_date, progress=False)
            if df.empty:
                continue

            df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
            df = df.reset_index()
            df.rename(columns={'Date': 'date', 'Open': 'open', 'High': 'high', 'Low': 'low',
                                'Close': 'close', 'Volume': 'volume'}, inplace=True)
            df['tic'] = ticker
            df = compute_complete_indicators(df, price_col='close')

            if USE_MULTI_TIMEFRAME:
                df = compute_multi_timeframe_features(df, price_col='close', date_col='date')

            processed_dfs.append(df)
        except Exception as e:
            print(f"❌ ไม่สามารถดาวน์โหลด {ticker}: {e}")

    if not processed_dfs:
        raise ValueError("ดาวน์โหลดข้อมูลล้มเหลว กรุณาตรวจสอบการเชื่อมต่ออินเทอร์เน็ต")

    final_df = pd.concat(processed_dfs, ignore_index=True)

    macro_tickers = {"^VIX": "vix", "^TNX": "bond_yield", "GC=F": "gold", "CL=F": "wti"}
    try:
        macro_raw = yf.download(list(macro_tickers.keys()), start=buffer_start, end=end_date, progress=False)
        macro_df = macro_raw['Close'].copy()
        macro_df.columns = [col[0] if isinstance(col, tuple) else col for col in macro_df.columns]
        macro_df.rename(columns=macro_tickers, inplace=True)
        macro_df = macro_df.reset_index().rename(columns={'Date': 'date'})
        final_df = pd.merge(final_df, macro_df, on='date', how='left')
    except Exception as e:
        for name in macro_tickers.values():
            final_df[name] = np.nan

    try:
        fed_funds = web.DataReader('FEDFUNDS', 'fred', buffer_start, end_date).reset_index()
        m2_supply = web.DataReader('M2SL', 'fred', buffer_start, end_date).reset_index()

        fed_funds.rename(columns={'DATE': 'date', 'FEDFUNDS': 'fed_rate'}, inplace=True)
        m2_supply.rename(columns={'DATE': 'date', 'M2SL': 'm2'}, inplace=True)

        fed_funds['date'] = pd.to_datetime(fed_funds['date']) + pd.Timedelta(days=FED_FUNDS_LAG_DAYS)
        m2_supply['date'] = pd.to_datetime(m2_supply['date']) + pd.Timedelta(days=M2_LAG_DAYS)

        final_df['date'] = pd.to_datetime(final_df['date'])
        final_df.sort_values('date', inplace=True)
        fed_funds.sort_values('date', inplace=True)
        m2_supply.sort_values('date', inplace=True)

        final_df = pd.merge_asof(final_df, fed_funds, on='date', direction='backward')
        final_df = pd.merge_asof(final_df, m2_supply, on='date', direction='backward')
    except Exception as e:
        print(f"⚠️ ดึงข้อมูล FRED ไม่สำเร็จ: {e}")
        final_df['fed_rate'] = np.nan
        final_df['m2'] = np.nan

    final_df.sort_values(['date', 'tic'], inplace=True)
    final_df = final_df.groupby('tic', group_keys=False).apply(lambda g: g.ffill())

    if USE_MULTI_TIMEFRAME:
        for col in MULTI_TIMEFRAME_INDICATORS:
            if col in final_df.columns:
                final_df[col] = final_df[col].fillna(0.0)

    final_df.dropna(inplace=True)

    final_df = final_df[final_df['date'] >= pd.to_datetime(start_date)].reset_index(drop=True)

    date_list = sorted(final_df['date'].unique())
    date2day = {date: day for day, date in enumerate(date_list)}
    final_df['day'] = final_df['date'].map(date2day)
    final_df['date'] = final_df['date'].dt.strftime('%Y-%m-%d')

    final_df = final_df.sort_values(['date', 'tic'])
    final_df.index = final_df['day'].values
    return final_df

def compute_macro_scaling_stats(train_df):
    scale_cols = ['m2', 'fed_rate', 'vix', 'bond_yield', 'gold', 'wti']
    stats = {}
    for col in scale_cols:
        if col in train_df.columns:
            mean = train_df[col].mean()
            std = train_df[col].std()
            std = std if std > 1e-8 else 1.0
            stats[col] = (mean, std)
    return stats

def apply_macro_scaling(df, stats):
    df = df.copy()
    for col, (mean, std) in stats.items():
        if col in df.columns:
            df[col] = (df[col] - mean) / std
    return df

In [6]:
# ==========================================
# 4. ROBUST CUSTOM ENVIRONMENT
# ==========================================
class RealisticTradingEnv(StockTradingEnv):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        if 'vix' in self.df.columns:
            self.vix_array = self.df.groupby(self.df.index)['vix'].first().sort_index().values
        else:
            self.vix_array = np.zeros(len(self.df.index.unique()))
        
        self._prev_cost = 0.0

    def reset(self, **kwargs):
        obs, info = super().reset(**kwargs)
        self._prev_cost = 0.0
        return obs, info

    def step(self, actions):
        state, reward, terminal, truncated, info = super().step(actions)
        if not terminal:
            reward = self._calculate_reward()
            self.reward = reward
            if len(self.rewards_memory) > 0:
                self.rewards_memory[-1] = reward
        return state, reward, terminal, truncated, info

    def _calculate_reward(self):
        current_portfolio_value = self.state[0] + sum(
            np.array(self.state[1:(self.stock_dim + 1)]) *
            np.array(self.state[(self.stock_dim + 1):(self.stock_dim * 2 + 1)])
        )

        # 🔧 แก้ไข #1 (Fatal Bug): เปลี่ยน [-1] เป็น [-2] เพื่อดึงมูลค่าพอร์ตของ "ตาที่แล้ว" มาเทียบกำไร
        if len(self.asset_memory) > 1:
            previous_portfolio_value = self.asset_memory[-2] 
        else:
            previous_portfolio_value = self.initial_amount
            
        if previous_portfolio_value <= 0:
            previous_portfolio_value = self.initial_amount

        step_return = (current_portfolio_value - previous_portfolio_value) / previous_portfolio_value
        
        cost_this_step = self.cost - self._prev_cost
        self._prev_cost = self.cost
        transaction_costs = cost_this_step * self.reward_scaling

        if step_return < 0:
            step_return *= 2.0

        current_vix = self.vix_array[self.day] if self.day < len(self.vix_array) else 0.0
        invested_ratio = 1 - (self.state[0] / current_portfolio_value) if current_portfolio_value > 0 else 0

        risk_penalty = 0.0
        if current_vix > 0.0:
            # 🔧 แก้ไข #5 (Latent Bug): เอาการหาร 10 ออก เพราะ current_vix ถูกทำ Z-score มาแล้ว
            vix_excess = current_vix 
            risk_penalty = vix_excess * invested_ratio * (abs(step_return) + 0.001)

        raw_reward = (step_return * 100) - transaction_costs - (risk_penalty * 100)
        reward = 10.0 * np.tanh(raw_reward / 10.0)

        return float(reward)

def get_env_kwargs(stock_dimension):
    state_space = 1 + (2 * stock_dimension) + (len(FEATURES) * stock_dimension)
    return {
        "hmax": 2000, # 🔧 แก้ไข #2 (Latent Bug): ปรับจาก 100 เป็น 2000 เพื่อลดปัญหา Cash Drag 
        "initial_amount": 1000000,
        "num_stock_shares": [0] * stock_dimension,
        "state_space": state_space,
        "stock_dim": stock_dimension,
        "tech_indicator_list": FEATURES,
        "action_space": stock_dimension,
        "reward_scaling": 1e-4,
        "buy_cost_pct": [TRANSACTION_COST_PCT] * stock_dimension,
        "sell_cost_pct": [TRANSACTION_COST_PCT] * stock_dimension,
    }

In [7]:
# ==========================================
# 5. TRANSLATION & REPORT ENGINE
# ==========================================
def explain_bot_performance(returns):
    print("\n" + "=" * 55)
    print("🤖 สรุปผลงานบอทเทรดในหน้าต่างทดสอบจริง (ฉบับเข้าใจง่าย)")
    print("=" * 55)

    total_return = qs.stats.comp(returns) * 100
    cagr = qs.stats.cagr(returns) * 100
    max_dd = qs.stats.max_drawdown(returns) * 100
    sharpe = qs.stats.sharpe(returns)
    win_rate = qs.stats.win_rate(returns) * 100

    print("\n💰 1. ด้านการทำกำไร (Return)")
    print(f"   • กำไรสะสมตลอดการทดสอบ: {total_return:.2f}%")
    print(f"   • ผลตอบแทนทบต้นเฉลี่ยรายปี (CAGR): {cagr:.2f}%")
    if cagr > 12:
        print("   ✅ สรุป: ยอดเยี่ยม! บอทสร้างผลงานชนะอัตราเฉลี่ยของตลาดส่วนใหญ่")
    elif cagr > 0:
        print("   ⚠️ สรุป: พอใช้ได้ พอร์ตโตขึ้นแต่ยังไม่โดดเด่นนัก")
    else:
        print("   ❌ สรุป: ล้มเหลว บอททำเงินต้นสูญหาย")

    print("\n📉 2. ด้านความเสี่ยง (Risk & Drawdown)")
    print(f"   • ช่วงที่พอร์ตร่วงหนักสุดจากจุดสูงสุด (Max Drawdown): {max_dd:.2f}%")
    if max_dd > -15:
        print("   ✅ สรุป: ปลอดภัยสูง ระบบคุมสัดส่วนการสูญเสียเงินได้ดีมาก")
    elif max_dd > -30:
        print("   ⚠️ สรุป: ปานกลาง พอร์ตแกว่งตามมาตรฐานสไตล์กองทุนเชิงรุก")
    else:
        print("   ❌ สรุป: อันตรายมาก! บอทปล่อยให้พอร์ตเสียหายหนักเกินไป")

    print("\n⚖️ 3. ความคุ้มค่าและสถิติการชนะ (Efficiency)")
    print(f"   • ความคุ้มค่าต่อหนึ่งหน่วยความเสี่ยง (Sharpe Ratio): {sharpe:.2f}")
    if sharpe >= 1.0:
        print("   ✅ สรุป: ดีเยี่ยม กำไรที่ได้คุ้มค่าอย่างมากกับความเสี่ยงที่ถือครอง")
    else:
        print("   ⚠️ สรุป: ความคุ้มค่าน้อยลง ผลตอบแทนอาจไม่สมน้ำสมเนื้อกับความเสี่ยงที่เผชิญ")
    print(f"   • อัตราความแม่นยำรายวัน (Win Rate): {win_rate:.2f}%")
    print("=" * 55 + "\n")

In [8]:
# ==========================================
# 6. HIGH-SPEED TRAINING PIPELINE WITH VALIDATION
# ==========================================
def train_model():
    TRAIN_START, TRAIN_END = '1999-04-01', '2019-12-31'
    VAL_START, VAL_END = '2020-01-01', '2023-12-31'

    os.makedirs("./exported_data", exist_ok=True)

    print(f"\nℹ️ Feature ทั้งหมดที่ใช้ ({len(FEATURES)} ตัว): {FEATURES}")
    print(f"ℹ️ Multi-timeframe: {'เปิดใช้งาน' if USE_MULTI_TIMEFRAME else 'ปิดอยู่ (ตั้ง USE_MULTI_TIMEFRAME=True เพื่อเปิด)'}")

    print("\n--- 🛠️ [1/3] เริ่มเตรียมข้อมูลสําหรับฝึกสอนบอท (Training Set) ---")
    train_df = fetch_and_prepare_data(TICKERS, TRAIN_START, TRAIN_END)

    print("\n--- 🛠️ [2/3] เริ่มเตรียมข้อมูลสําหรับข้อสอบ (Validation Set) ---")
    val_df = fetch_and_prepare_data(TICKERS, VAL_START, VAL_END)

    scaling_stats = compute_macro_scaling_stats(train_df)
    train_df = apply_macro_scaling(train_df, scaling_stats)
    val_df = apply_macro_scaling(val_df, scaling_stats)

    with open("./exported_data/macro_scaling_stats.json", "w") as f:
        json.dump({k: list(v) for k, v in scaling_stats.items()}, f)

    train_df.to_csv("./exported_data/train_set.csv", index=False)
    val_df.to_csv("./exported_data/validation_set.csv", index=False)
    print("📁 [Exported] บันทึกไฟล์ Training/Validation Set และสถิติ Scaling เรียบร้อย")

    env_kwargs = get_env_kwargs(len(TICKERS))

    def make_env():
        return lambda: RealisticTradingEnv(df=train_df, **env_kwargs)

    NUM_CPU = 4
    env_train = SubprocVecEnv([make_env() for _ in range(NUM_CPU)])
    env_val = DummyVecEnv([lambda: RealisticTradingEnv(df=val_df, **env_kwargs)])

    os.makedirs("./best_model", exist_ok=True)
    eval_callback = EvalCallback(
        env_val,
        best_model_save_path='./best_model/',
        log_path='./best_model/logs/',
        eval_freq=max(500, 2048 // NUM_CPU),
        deterministic=True,
        render=False
    )

    # ภายในฟังก์ชัน train_model() เปลี่ยนโค้ดส่วนการสร้าง agent เป็นแบบนี้:
    
    policy_kwargs = dict(
        lstm_hidden_size=128,  # ขนาดของหน่วยความจำ (ปรับเป็น 256 ได้ถ้าอยากให้จำเก่งขึ้น แต่เทรนช้าลง)
        n_lstm_layers=1,       # จำนวนชั้นของเครือข่าย LSTM
    )

    agent = RecurrentPPO(
        "MlpLstmPolicy",       # ใช้ Policy ที่มี LSTM แฝงอยู่
        env_train,
        learning_rate=0.00025,
        n_steps=2048,
        batch_size=256,
        policy_kwargs=policy_kwargs, # ใส่ตั้งค่า LSTM เข้าไป
        device=device,
        verbose=1
    )

    print("\n--- 🚀 [3/3] บอทเริ่มกระโจนเข้าสู่การเรียนรู้แบบคู่ขนาน ---")
    try:
        agent.learn(total_timesteps=300_000, callback=eval_callback)
        agent.save("ppo_realistic_trading_bot_last")
        print("💾 บันทึกโมเดลเสร็จสิ้น! บอทเวอร์ชันที่ดีที่สุดถูกบรรจุอยู่ในโฟลเดอร์ './best_model/'")
    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาดในจังหวะเทรนโมเดล: {e}")

In [9]:
# ==========================================
# 7. COMPREHENSIVE OUT-OF-SAMPLE TEST
# ==========================================
def test_model():
    TEST_START, TEST_END = '2024-01-01', '2026-07-29'

    print("\n🔮 ดึงข้อมูลตลาดปัจจุบันมาสับไพ่ทดสอบจริง (Out-of-Sample Test)...")
    test_df = fetch_and_prepare_data(TICKERS, TEST_START, TEST_END)

    with open("./exported_data/macro_scaling_stats.json", "r") as f:
        scaling_stats = {k: tuple(v) for k, v in json.load(f).items()}
    test_df = apply_macro_scaling(test_df, scaling_stats)

    os.makedirs("./exported_data", exist_ok=True)
    test_df.to_csv("./exported_data/test_set.csv", index=False)
    print("📁 [Exported] บันทึกไฟล์ Test Set เรียบร้อย -> ./exported_data/test_set.csv")

    model_path = "./best_model/best_model.zip"
    if not os.path.exists(model_path):
        model_path = "ppo_realistic_trading_bot_last"
        print("⚠️ ไม่พบโมเดลคัดสรรพิเศษยามทำลายสถิติ วนกลับไปใช้โมเดลรอบสุดท้าย")

    # ภายในฟังก์ชัน test_model() เปลี่ยนบล็อกการโหลดโมเดลและลูปเทรด เป็นแบบนี้:

    trained_model = RecurrentPPO.load(model_path, device=device)

    env_kwargs = get_env_kwargs(len(TICKERS))
    e_test_gym = RealisticTradingEnv(df=test_df, **env_kwargs)
    env_test = DummyVecEnv([lambda: e_test_gym])

    obs = env_test.reset()
    account_memory = []
    num_days = len(test_df.index.unique())

    # --- ส่วนที่ต้องเพิ่มสำหรับ LSTM ---
    lstm_states = None  # เริ่มต้น episode ความจำจะเป็นว่างเปล่า
    # สถานะว่าเป็นการเริ่มรอบใหม่หรือไม่ (LSTM จะได้ลบความจำเดิมทิ้งตอนเริ่มรันใหม่)
    episode_starts = np.ones((1,), dtype=bool) 
    
    print("📈 บอทกำลังดำเนินการจำลองการกระจายสินทรัพย์จริงในอดีต (โหมด LSTM)...")
    for i in range(num_days):
        # 1. โยน obs พร้อมความจำล่าสุด (lstm_states) ให้โมเดลคิด
        action, lstm_states = trained_model.predict(
            obs,
            state=lstm_states,
            episode_start=episode_starts,
            deterministic=True
        )
        
        # 2. นำ Action ไปรันในตลาดจริง
        obs, rewards, dones, info = env_test.step(action)
        
        # 3. อัปเดต episode_starts ถ้า dones เป็น True บอทจะรีเซ็ตความจำอัตโนมัติในตาถัดไป
        episode_starts = dones 

        if i == num_days - 2:
            account_memory = env_test.env_method(method_name="save_asset_memory")[0]

        if dones[0]:
            print("🏁 การทดสอบย้อนหลังสิ้นสุดสมบูรณ์!")
            break

    df_account_value = pd.DataFrame(account_memory)
    df_account_value['date'] = pd.to_datetime(df_account_value['date'])
    df_account_value.set_index('date', inplace=True)

    df_account_value['daily_return'] = df_account_value['account_value'].pct_change()
    df_account_value.dropna(inplace=True)

    explain_bot_performance(df_account_value['daily_return'])



In [10]:
# ==========================================
# 8. EXECUTION GATEWAY
# ==========================================
if __name__ == "__main__":
    train_model()
    test_model()


ℹ️ Feature ทั้งหมดที่ใช้ (20 ตัว): ['RSI_14', 'RSI_signal', 'RSI_overbought', 'RSI_oversold', 'RSI_above_center', 'MACD_histogram', 'MACD_cross', 'price_to_EMA50_ratio', 'price_to_EMA200_ratio', 'EMA_golden_cross', 'EMA_death_cross', 'StochRSI_K', 'StochRSI_D', 'StochRSI_cross', 'vix', 'bond_yield', 'gold', 'wti', 'fed_rate', 'm2']
ℹ️ Multi-timeframe: ปิดอยู่ (ตั้ง USE_MULTI_TIMEFRAME=True เพื่อเปิด)

--- 🛠️ [1/3] เริ่มเตรียมข้อมูลสําหรับฝึกสอนบอท (Training Set) ---
📦 กําลังดาวน์โหลดข้อมูลจาก 1999-04-01 ถึง 2019-12-31 (ดึงเผื่อ Buffer ตั้งแต่ 1996-12-02)...

--- 🛠️ [2/3] เริ่มเตรียมข้อมูลสําหรับข้อสอบ (Validation Set) ---
📦 กําลังดาวน์โหลดข้อมูลจาก 2020-01-01 ถึง 2023-12-31 (ดึงเผื่อ Buffer ตั้งแต่ 2017-09-03)...
📁 [Exported] บันทึกไฟล์ Training/Validation Set และสถิติ Scaling เรียบร้อย
Using cpu device

--- 🚀 [3/3] บอทเริ่มกระโจนเข้าสู่การเรียนรู้แบบคู่ขนาน ---
Eval num_timesteps=2048, episode_reward=0.00 +/- 0.00
Episode length: 1006.00 +/- 0.00
---------------------------------
| e